In [ ]:
import numpy as np
import mediapy as media
import sys
import os
import pandas as pd
from brian2 import *

sys.path.append(os.path.abspath('./fly-brain/code'))

print("Building brain...")
import benchmark

class DigitalBrain:
    def __init__(self, weights_path):
        self.weights_path = weights_path

        eqs = '''
        dv/dt = (v_rest - v) / tau : volt
        v_rest : volt
        tau : second
        '''

        self.neuron_count = 139255
        self.neurons = NeuronGroup(self.neuron_count, eqs, threshold='v > -50*mV', reset='v = -70*mV', method='exact')
        self.neurons.v = -70*mV
        self.neurons.v_rest = -70*mV
        self.neurons.tau = 20*ms

        self.sensory_group = PoissonGroup(100, rates=0*Hz)
        self.sensory_synapses = Synapses(self.sensory_group, self.neurons, on_pre='v_post += 5*mV')
        self.sensory_synapses.connect(j='i')

        self.motor_monitor = SpikeMonitor(self.neurons)

        self.net = Network(self.neurons, self.sensory_group, self.motor_monitor)

    def load_model_data(self):
        if os.path.exists(self.weights_path):
            df = pd.read_parquet(self.weights_path)

            self.synapses = Synapses(self.neurons, self.neurons, 'w : volt', on_pre='v_post += w')

            self.synapses.connect(i=df['Presynaptic_Index'].to_numpy(), j=df['Postsynaptic_Index'].to_numpy())
            
            self.synapses.w = df['Excitatory x Connectivity'].to_numpy() * 0.1 * mV

            self.net.add(self.synapses)
            
            print("54.5M Synapses mapped.")
        else:
            print("Error .parquet file not found.")

    def simulate_step(self, sensory_input):
        self.sensory_group.rates = sensory_input * Hz
        start_count = len(self.motor_monitor.i)
        self.net.run(20*ms)
        return self.motor_monitor.i[start_count:]
        
data_path = 'PATH_TO_PARQUET_FILE_HERE'
brain = DigitalBrain(data_path)
brain.load_model_data()

print("Brain fully loaded.")

In [ ]:
from flybody.fly_envs import walk_imitation

env = walk_imitation()
timestep = env.reset()
print("Body ready in MuJoCo physeng.")

In [ ]:
import torch

class SensorimotorBridge:
    
        def __init__(self, brain, env):
            
            self.brain = brain
            self.env = env
            
            
            self.motor_neuron_ids = np.arange(139000, 139059)
            self.action_dim = 59

        def step(self, observation):
            
            sensory_input = self.encode_sensors(observation)

            spikes = self.brain.simulate_step(sensory_input)
            return self.decode_spikes(spikes)

        def encode_sensors(self, obs):
            flat_obs = None
            
            if isinstance(obs, dict):
                val_list = [np.asarray(v).flatten() for v in obs.values() if v is not None]
                if val_list:
                    flat_obs = np.concatenate(val_list)
                else:
                    flat_obs = np.zeros(100)
            elif obs is not None:
                flat_obs = np.asarray(obs).flatten()
            else:
                flat_obs = np.zeros(100)

            flat_obs = np.nan_to_num(np.array(flat_obs, dtype=float))

            flat_obs = np.resize(flat_obs, (100,))
            return np.clip(flat_obs, 0, 1) * 200

        def decode_spikes(self, spikes):
            actions = np.zeros(self.action_dim)
            
            for i, nid in enumerate(self.motor_neuron_ids):
                
                if nid in spikes:

                    actions[i] = 1.0

            return actions

bridge = SensorimotorBridge(brain, env)

print("Phys sim complete.")

In [ ]:
import mediapy as media

timestep = env.reset()
video_frames = [ ]

for step in range(250):
    obs = timestep.observation

    neural_action = bridge.step(obs)

    timestep = env.step(neural_action)

    if timestep.last():
        break

    pixels = env.physics.render(height=480, width=640, camera_id = 1)
    video_frames.append(pixels)

media.show_video(video_frames, fps=30)